Step 1: Creating Tokens

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

print("Total chars: ", len(raw_text))
print(raw_text[:100])

Total chars:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [ ]:
import re

text = "Hey, This is Darsh!! How's it's going??"
res = re.split(r'(\s)', text)

print(res)

['Hey,', ' ', 'This', ' ', 'is', ' ', 'Darsh!!', ' ', "How's", ' ', "it's", ' ', 'going??']


In [ ]:
res = re.split(r'([,.]|\s)', text)

print(res)

['Hey', ',', '', ' ', 'This', ' ', 'is', ' ', 'Darsh!!', ' ', "How's", ' ', "it's", ' ', 'going??']


In [ ]:
res = [item for item in res if item.strip()]
print(res)

['Hey', ',', 'This', 'is', 'Darsh!!', "How's", "it's", 'going??']


In [ ]:
# To remove the white space
text = "Hello, World. Is this-- a test?"
res = re.split(r'([,.:;?_!"()\']|--|\s)', text)
res = [item.strip() for item in res if item.strip()]
print(res)

['Hello', ',', 'World', '.', 'Is', 'this', '--', 'a', 'test', '?']


Now we'll apply the same Regex we applied above on the raw text from the book

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [ ]:
print(len(preprocessed))

4690


Step 2 : Creating the Token Ids

In [ ]:
words = sorted(set(preprocessed))
vocab_size = len(words)
print(vocab_size)

1130


In [ ]:
vocab = {token: integer for integer, token in enumerate(words)}

In [ ]:
for i, itm in enumerate(vocab.items()):
  print(itm)
  if i >= 50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [ ]:
class Tokenizer:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i : s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]

    ids = [self.str_to_int[item] for item in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[item] for item in ids])
    # Replace spaces before the specified punctuations
    # re.sub(pattern, replacement, text)
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer = Tokenizer(vocab)
text = """"It's the last he painted, you know,"
          Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [ ]:
tokenizer.decode(ids=ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [ ]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

# **Adding Special context Tokens**

For this make another class which will handle unknown words



In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

1132

In [ ]:
for i, itms in enumerate(list(vocab.items())[-5:]):
  print(itms)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [ ]:
class TokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i : s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]

    preprocessed = [
        item if item in self.str_to_int else "<|unk|>" for item in preprocessed
    ]

    ids = [self.str_to_int[item] for item in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[item] for item in ids])
    # Replace spaces before the specified punctuations
    # re.sub(pattern, replacement, text)
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer = TokenizerV2(vocab)
text_1 = "Hello, do you like tea?"
text_2 = "No, I have one"

text = "<|endoftext|> ".join((text_1, text_2))
print(text)

Hello, do you like tea?<|endoftext|> No, I have one


In [ ]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 70, 5, 53, 530, 729]

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> No, I have one'